## 1. ¿Qué es Seaborn?

Seaborn es una biblioteca de visualización estadística construida sobre Matplotlib y diseñada para trabajar cómodamente con DataFrames.

**Ventajas**
- Sintaxis declarativa: indicamos `data`, `x`, `y`, `hue`.
- Visualizaciones estadísticas: distribuciones, relaciones y comparaciones.
- Paletas, leyendas y estilos integrados.

### Sintaxis general
```python
sns.scatterplot(data=df, x="variable_x", y="variable_y", hue="grupo")
```

| Parámetro | Significado |
|---|---|
| `data` | DataFrame |
| `x`, `y` | Variables de los ejes |
| `hue` | Codifica categorías mediante color |
| `size` | Codifica magnitud mediante tamaño |
| `style` | Codifica grupos mediante marcador |
| `palette` | Paleta de colores |
| `ax` | Eje Matplotlib donde dibujar |

Seaborn crea la gráfica; Matplotlib permite ajustar títulos, etiquetas, límites y disposición.

## 2. Fundamentos teóricos: ¿qué aporta Seaborn?
Seaborn es una interfaz de alto nivel construida sobre Matplotlib. Facilita gráficos estadísticos, asignación semántica de variables (`hue`, `row`, `col`), estimaciones agregadas e intervalos de confianza.

| Función | Pregunta típica |
|---|---|
| `histplot` / `displot` | ¿Cómo se distribuye una variable? |
| `boxplot` / `violinplot` | ¿Cómo varía una variable entre grupos? |
| `barplot` | ¿Cómo se compara una estadística agregada? |
| `scatterplot` / `regplot` | ¿Existe asociación entre dos variables? |
| `heatmap` | ¿Cómo se comporta una matriz de valores? |

Un gráfico correcto no demuestra causalidad. La selección del estimador (`mean`, `median`, `sum`) debe responder a la pregunta y al significado de la variable.

## 3. Preparación del entorno y carga

Instalación:
```bash
pip install pandas numpy matplotlib seaborn openpyxl
```



### Sintaxis fundamental

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# DataFrame de ejemplo para la gráfica
df = pd.DataFrame({
    "demanda_diaria": [120, 85, 200, 45, 150, 90, 210, 60, 130, 75, 180, 100],
    "existencias": [30, 120, 25, 200, 50, 140, 20, 180, 70, 110, 40, 95],
    "nivel_riesgo": ["Alto", "Medio", "Alto", "Bajo", "Alto", "Medio", "Alto", "Bajo", "Medio", "Medio", "Alto", "Medio"],
    "venta_neta": [15000, 8000, 22000, 4000, 17000, 9000, 25000, 5000, 12000, 7500, 19000, 10000]
})

sns.scatterplot(
    data=df,
    x="demanda_diaria",
    y="existencias",
    hue="nivel_riesgo",
    size="venta_neta"
)

plt.title("Demanda diaria vs existencias")
plt.xlabel("Demanda diaria")
plt.ylabel("Existencias")
plt.show()

| Parámetro | Función |  
|---|---|  
| `data` | DataFrame de origen |  
| `x` | Variable del eje horizontal |  
| `y` | Variable del eje vertical |  
| `hue` | Variable que determina el color |  
| `size` | Variable que determina el tamaño |  
| `style` | Variable que modifica el marcador |  
| `palette` | Paleta de colores |  
| `ax` | Eje de Matplotlib donde se dibuja |

# Seaborn aplicado al diagnóstico exploratorio de riesgo de abastecimiento


## Caso del socio formador
La empresa busca reducir situaciones de posible falta de producto y contar con evidencia para priorizar revisiones. Construiremos un **semáforo exploratorio** a partir de ventas recientes y una fotografía de inventario.

> **Alcance:** esto no es un sistema de compras ni una predicción validada de desabasto. Los umbrales de 15 y 30 días son didácticos. No incorporan lead time, pedidos abiertos, stock de seguridad, inventario comprometido, estacionalidad ni políticas comerciales.

### Preguntas guía
1. ¿Qué proporción de SKU con ventas recientes tiene correspondencia en inventario?
2. ¿Cómo se distribuye la cobertura estimada?
3. ¿Qué productos combinan cobertura corta y mayor actividad comercial?
4. ¿Cómo cambia la composición de niveles exploratorios entre marcas?
5. ¿Qué falta validar antes de usar esto operativamente?

### Conceptos
- **Ventas recientes:** unidades registradas en la ventana temporal.
- **Demanda diaria observada:** unidades vendidas / días de la ventana.
- **Existencias al corte:** fotografía de inventario en una fecha.
- **Cobertura estimada:** existencias al corte / demanda diaria observada.

La cobertura es una razón descriptiva, no una métrica exacta de cuánto durará el inventario.

## 4. Preparación y carga de fuentes

Se utilizan dos archivos:
- **Workbook principal:** contiene la hoja de ventas.
- **Workbook complementario:** contiene `Existencias_15_10_2026` y `Histórico_Entradas_SAlidas`.

Los movimientos solo se usan para auditar `clave_articulo` ↔ `SKU`. No se incorporan a la métrica de cobertura del inventario.

Coloca ambos Excel en una carpeta `Datos` junto al notebook o ajusta las rutas.

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook", palette="deep") # Establece un fondo blanco con una cuadrícula (líneas guía) gris claro.
# Ajusta automáticamente la escala de los elementos del gráfico para que se vean proporcionados dentro de un entorno como Jupyter Notebook
pd.set_option("display.max_columns", 40) #• Evita que pandas oculte columnas intermedias con puntos suspensivos (...)
#si tu tabla tiene muchas variables. Permite ver hasta 40 columnas completas a la vez.
pd.set_option("display.float_format", lambda x: f"{x:,.2f}") # Configura pandas para mostrar los números de punto flotante con 
#dos decimales y separadores de miles, lo que mejora la legibilidad de los datos numéricos.


PATH_VENTAS = Path.cwd() / 'Datos' / 'Datos_CLV_2026_Tec_v2.xlsx'
PATH_COMPLEMENTARIO = Path.cwd()/ 'Datos' / 'Datos_Complementarios_CLV.xlsx'

OUT_DIR = Path.cwd() / "salidas_seaborn_abasto"
OUT_DIR.mkdir(parents=True, exist_ok=True)


ventas = pd.read_excel(PATH_VENTAS, sheet_name="Ventas_CLV_2022_2026")
existencias = pd.read_excel(PATH_COMPLEMENTARIO, sheet_name="Existencias_15_10_2026")
salidas = pd.read_excel(PATH_COMPLEMENTARIO, sheet_name="Histórico_Entradas_SAlidas")

print("Ventas:", ventas.shape)
print("Existencias:", existencias.shape)
print("Movimientos:", salidas.shape)

In [ ]:
display(ventas.head(3))
display(existencias.head(3))
display(salidas.head(3))
print('Columnas ventas:', ventas.columns.tolist())
print('Columnas existencias:', existencias.columns.tolist())
print('Columnas movimientos:', salidas.columns.tolist())


### Granularidad

Antes de unir tablas:
- ¿Una fila de ventas representa un artículo, una transacción o ambas cosas?
- ¿Una fila de existencias representa una transacción o una fotografía del stock?
- ¿Qué ocurre si una tabla tiene varias filas por ID y se une directamente con otra que también las repite?

**Regla:** primero agregamos las ventas a nivel SKU; después hacemos el `merge` contra inventario agregado al mismo nivel.



La llave `articulo_id` ↔ `articulo` se usa para relacionar ventas e inventario en este ejercicio. La normalización solo corrige diferencias de formato; la equivalencia de catálogo debe confirmarse con el socio formador.

In [ ]:
def normalizar_llave(s):
    # Normalizar formato no demuestra equivalencia semántica entre catálogos.
    return (
        s.astype("string").str.strip().str.upper()
        .str.replace(r"\.0$", "", regex=True)
        .replace({"": pd.NA, "NAN": pd.NA, "NONE": pd.NA, "<NA>": pd.NA})
    )

def convertir_fecha(serie):
    # Admite fechas Excel/datetime y cadenas con formatos habituales.
    return pd.to_datetime(serie, errors="coerce", dayfirst=True)

ventas["fecha_documento"] = convertir_fecha(ventas["fecha_documento"])
ventas["sku_inventario"] = normalizar_llave(ventas["articulo_id"])
ventas["sku_salida"] = normalizar_llave(ventas["clave_articulo"])
existencias["sku_inventario"] = normalizar_llave(existencias["articulo"])
salidas["sku_salida"] = normalizar_llave(salidas["SKU"])

for col in ["total_unidades", "total_neto"]:
    ventas[col] = pd.to_numeric(ventas[col], errors="coerce")
for col in ["existencias", "valor_inventario"]:
    existencias[col] = pd.to_numeric(existencias[col], errors="coerce")

print("Fechas válidas:", f"{ventas['fecha_documento'].notna().mean():.1%}")
print("Rango:", ventas["fecha_documento"].min(), "a", ventas["fecha_documento"].max())
print("Llaves nulas ventas:", ventas["sku_inventario"].isna().sum())
print("Llaves nulas inventario:", existencias["sku_inventario"].isna().sum())

In [ ]:
def cobertura_llaves(izq, der, etiqueta):
    # set(...) convierte cada columna en un conjunto matemático, lo que permite compararlos muy rápido

    izq, der = set(izq.dropna().unique()), set(der.dropna().unique())
    # ambos = izq & der: Utiliza el operador de intersección (&) para encontrar los SKUs que existen en ambas columnas al mismo tiempo
    ambos = izq & der
    return pd.Series({
        "comparación": etiqueta,
        "llaves_izquierda": len(izq),
        "llaves_derecha": len(der),
        "coincidencias": len(ambos), #Cantidad de SKUs que lograron empatar entre ambas tablas
        "cobertura_izquierda_%": round(100 * len(ambos) / max(len(izq), 1), 2), #Porcentaje de los SKUs de cada tabla que tienen correspondencia en la otra
        "cobertura_derecha_%": round(100 * len(ambos) / max(len(der), 1), 2), #Porcentaje de los SKUs de cada tabla que tienen correspondencia en la otra
    })

auditoria = pd.DataFrame([
    cobertura_llaves(ventas["sku_inventario"], existencias["sku_inventario"],
                     "Ventas.articulo_id ↔ Existencias.articulo"),
    cobertura_llaves(ventas["sku_salida"], salidas["sku_salida"],
                     "Ventas.clave_articulo ↔ Movimientos.SKU"),
])
display(auditoria)
print("Duplicados llave inventario:", existencias["sku_inventario"].duplicated().sum())
print("Duplicados llave movimientos:", salidas["sku_salida"].duplicated().sum())

### Pausa de interpretación

**¿Qué significa cobertura izquierda?** Porcentaje de llaves únicas del conjunto izquierdo que también aparecen en el derecho. En la primera fila, representa qué proporción de SKU observados en ventas encuentra correspondencia en el inventario.

**¿Una cobertura alta prueba que todos los movimientos o la demanda están completos?** No. Solo mide coincidencia de identificadores únicos, no completitud de transacciones, unidades, fechas o registros.



## 5. Construir una base de análisis: ventas recientes + inventario

Se selecciona una ventana móvil de 180 días, anclada a la fecha máxima válida de ventas. La ventana debe ser suficientemente reciente y representativa para el propósito del análisis.

**Pregunta:** ¿por qué no usar todo el histórico?  


**Advertencia sobre unidades:** si existen devoluciones, ajustes o unidades negativas, deben entenderse antes de interpretar la suma como demanda. La métrica aquí es actividad neta registrada en la columna `total_unidades`, no demanda censurada ni pedidos no satisfechos.

In [ ]:
DAYS = 180  # prueba 90 o 365.
fecha_corte_ventas = ventas["fecha_documento"].max()
if pd.isna(fecha_corte_ventas):
    raise ValueError("No hay fechas válidas; revisa el formato de fecha.")

fecha_inicio = fecha_corte_ventas - pd.Timedelta(days=DAYS)
ventas_recientes = ventas.loc[
    ventas["fecha_documento"].between(fecha_inicio, fecha_corte_ventas)
].copy()

print(f"Ventana: {fecha_inicio.date()} a {fecha_corte_ventas.date()} ({DAYS} días)")
print("Filas:", len(ventas_recientes))
print("Unidades nulas:", ventas_recientes["total_unidades"].isna().sum())
print("Unidades negativas:", (ventas_recientes["total_unidades"] < 0).sum())

**Consolidar (agregar)** ambas tablas a nivel de SKU único antes de unirlas.

Si un SKU aparece 50 veces en ventas y 3 veces en existencias (por estar en distintos bases de datos), unirlas directamente multiplicaría las filas (50 × 3 = 150), duplicando falsamente tanto las ventas como el inventario. Agrupar primero garantiza una relación limpia 1 a 1.

In [ ]:
# Agregar ventas primero: una fila por SKU evita una unión many-to-many.
demanda = (
    ventas_recientes.dropna(subset=["sku_inventario"])
    .groupby("sku_inventario", as_index=False)
    .agg(
        nombre_articulo=("nombre_articulo", "first"),
        marca=("marca", "first"),
        unidades_vendidas=("total_unidades", "sum"),
        venta_neta=("total_neto", "sum"),
        folios=("folio", "nunique"),
        clientes=("nombre_cliente", "nunique"),
        registros_venta=("sku_inventario", "size"),
    )
)
demanda["demanda_diaria"] = demanda["unidades_vendidas"] / DAYS

inventario = (
    existencias.dropna(subset=["sku_inventario"])
    .groupby("sku_inventario", as_index=False)
    .agg(
        existencias=("existencias", "sum"),
        valor_inventario=("valor_inventario", "sum"),
        almacenes=("almacen", "nunique"),
    )
)
print("SKU en demanda:", demanda["sku_inventario"].nunique())
print("SKU en inventario agregado:", inventario["sku_inventario"].nunique())
print("Llaves únicas:", demanda["sku_inventario"].is_unique, inventario["sku_inventario"].is_unique)

In [ ]:
demanda 

In [ ]:
inventario

**Análisis**: 

Al terminar este bloque, tenemos dos tablas perfectamente alineadas (demanda e inventario), ambas con un único registro por sku_inventario, listas para cruzarse con un pd.merge() sin riesgo de inflar cifras y calcular indicadores como días de inventario o riesgo de desabasto.

In [ ]:
riesgo = demanda.merge(
    inventario, on="sku_inventario", how="left",
    indicator="estado_union", validate="one_to_one"
)

#Auditoría post-cruce
display(
    riesgo["estado_union"].value_counts()
    .rename_axis("estado_union").reset_index(name="skus")
)
print("Cobertura de SKU de demanda con inventario:",
      f"{riesgo['estado_union'].eq('both').mean():.2%}")

#Limpieza de nulos sin perder trazabilidad
riesgo["match_inventario"] = riesgo["estado_union"].eq("both") #Guarda una bandera booleana (True/False) antes de rellenar los ceros.
print(riesgo["match_inventario"].head(3))
# Los nulos se sustituyen para cálculo, pero el indicador impide confundir
# ausencia de correspondencia con un stock cero confirmado.
riesgo["existencias"] = riesgo["existencias"].fillna(0)
riesgo["valor_inventario"] = riesgo["valor_inventario"].fillna(0)
riesgo["almacenes"] = riesgo["almacenes"].fillna(0).astype(int)
print("SKU sin correspondencia:", (~riesgo["match_inventario"]).sum())

## 6. Métrica exploratoria: cobertura y semáforo

La fórmula es:

$$
\text{Demanda diaria observada} = \frac{\text{Unidades vendidas en la ventana}}{\text{Días de la ventana}}
$$

$$
\text{Cobertura estimada (días)} = \frac{\text{Existencias al corte}}{\text{Demanda diaria observada}}
$$

### Umbrales didácticos
- **Crítico:** cobertura menor a 15 días.
- **Vigilar:** cobertura desde 15 y menor a 30 días.
- **Saludable (regla didáctica):** cobertura de 30 días o más.
- **Sin ventas observadas:** no se calcula cobertura porque la demanda diaria observada no es positiva.
- **Revisar catálogo:** no se encontró correspondencia de inventario para el SKU.

Los umbrales son parámetros para practicar clasificación y visualización. El socio formador debe validarlos contra lead time, pedidos abiertos, stock de seguridad, inventario comprometido y políticas por producto.

**Pregunta:** ¿por qué “sin ventas observadas” no debe etiquetarse como saludable?  

**Pregunta:** ¿por qué no se convierte automáticamente un SKU sin match en stock cero?  


**Este bloque** representa el motor de decisión del análisis: calcula cuántos días durará el inventario actual de cada producto y clasifica todo el catálogo en categorías accionables según su nivel de riesgo de desabasto.

In [ ]:
UMBRAL_CRITICO = 15
UMBRAL_VIGILAR = 30

#Solo se calcula si el producto tiene ventas positivas (demanda_diaria > 0) 
# Y si cruzó correctamente con la tabla de inventario (match_inventario == True).
# Si se cumple: Divide existencias / demanda_diaria (ej. si tienes 30 piezas y vendes 3 al día, tienes 10.0 días de cobertura).
riesgo["cobertura_dias"] = np.where(
    (riesgo["demanda_diaria"] > 0) & riesgo["match_inventario"],
    riesgo["existencias"] / riesgo["demanda_diaria"],
    np.nan,
)

def clasificar_riesgo(fila):
    if not fila["match_inventario"]:
        return "Revisar catálogo"
    if fila["demanda_diaria"] <= 0:
        return "Sin ventas observadas" #Separa artículos que sí existen pero no tuvieron rotación positiva en el periodo
    if fila["cobertura_dias"] < UMBRAL_CRITICO:
        return "Crítico" #Separa artículos que sí existen pero no tuvieron rotación positiva en el periodo
    if fila["cobertura_dias"] < UMBRAL_VIGILAR:
        return "Vigilar" #Productos en zona amarilla que aún tienen inventario pero empiezan a acercarse al punto de reorden.
    return "Saludable (regla didáctica)" #Si pasó todos los filtros anteriores, significa que tiene inventario suficiente por encima del umbral de vigilancia.


#Ordenamiento categórico y tabla resumen
riesgo["nivel_riesgo"] = riesgo.apply(clasificar_riesgo, axis=1)
orden_riesgo = [
    "Crítico", "Vigilar", "Saludable (regla didáctica)",
    "Sin ventas observadas", "Revisar catálogo"
]
riesgo["nivel_riesgo"] = pd.Categorical(
    riesgo["nivel_riesgo"], categories=orden_riesgo, ordered=True
)
resumen_riesgo = (
    riesgo["nivel_riesgo"].value_counts().reindex(orden_riesgo, fill_value=0)
    .rename_axis("nivel_riesgo").reset_index(name="numero_sku")
)
resumen_riesgo["porcentaje"] = (
    100 * resumen_riesgo["numero_sku"] / max(len(riesgo), 1)
).round(1)
display(resumen_riesgo)

In [ ]:
tabla_riesgo = riesgo.sort_values(
    ["nivel_riesgo", "venta_neta"], ascending=[True, False]
).copy()
columnas_tabla = [
    "sku_inventario", "nombre_articulo", "marca", "unidades_vendidas",
    "venta_neta", "folios", "existencias", "cobertura_dias",
    "nivel_riesgo", "match_inventario"
]
display(tabla_riesgo[columnas_tabla].head(20))

ruta_tabla = OUT_DIR / "tabla_riesgo_abastecimiento.csv"
tabla_riesgo.to_csv(ruta_tabla, index=False, encoding="utf-8-sig")
print("Tabla exportada:", ruta_tabla.resolve())

# 7. Visualizaciones Seaborn

Las gráficas se ordenan desde la vista general hacia la priorización de productos.

## Gráfica 1 — ¿Cómo se distribuye la cobertura?
Usamos `histplot` para observar concentración, asimetría y valores extremos. Para evitar que coberturas muy altas aplasten la lectura, se visualiza un máximo de 180 días, sin modificar el dato original.

### Histograma apilado y codificado por colores 


In [ ]:
CAP_GRAFICA = 180  # Solo limita la visualización, no modifica el dato original.

#Selecciona únicamente los SKUs que sí cruzaron con inventario (match_inventario)
#  y que tienen un cálculo válido de días de cobertura (.notna())

hist_df = tabla_riesgo.loc[
    tabla_riesgo["match_inventario"] & tabla_riesgo["cobertura_dias"].notna(),
    ["cobertura_dias", "nivel_riesgo"]
].copy()

#Este es un detalle clave de visualización. En inventarios reales, suele haber productos con 1,000 o 2,000
# días de stock (sobreinventario), lo que aplastaría todas las barras importantes cerca del cero. .clip(upper=180) 
# toma cualquier valor mayor a 180 días y lo agrupa en la última barra (180), manteniendo intacta la zona crítica 
# donde se toman decisiones.

hist_df["cobertura_grafica"] = hist_df["cobertura_dias"].clip(upper=CAP_GRAFICA)

fig, ax = plt.subplots(figsize=(11, 5)) #Define un lienzo panorámico (11 pulgadas de ancho por 5 de alto).
sns.histplot(
    data=hist_df, x="cobertura_grafica", bins=36, hue="nivel_riesgo", #Divide el rango de 0 a 180 días en 36 intervalos iguales.
    # Colorea y apila las barras según la clasificación de riesgo.
    multiple="stack",
    hue_order=["Crítico", "Vigilar", "Saludable (regla didáctica)"],
    palette={"Crítico": "#C62828", "Vigilar": "#F9A825",
             "Saludable (regla didáctica)": "#2E7D32"},
    ax=ax,
)
ax.axvline(UMBRAL_CRITICO, color="#C62828", linestyle="--", # Dibuja líneas verticales punteadas (linestyle="--")
           label=f"Crítico: {UMBRAL_CRITICO} días")
ax.axvline(UMBRAL_VIGILAR, color="#F9A825", linestyle="--",
           label=f"Vigilar: {UMBRAL_VIGILAR} días")
ax.set(title="Distribución de cobertura estimada por SKU",
       xlabel=f"Cobertura estimada (días; tope visual {CAP_GRAFICA})",
       ylabel="Número de SKU")
ax.legend()
plt.tight_layout() #Ajusta automáticamente los márgenes para que no se corten los textos y renderiza la gráfica en pantalla.

plt.show()

### Cómo leer el histograma — interpretación 

`histplot` agrupa los SKU en intervalos de cobertura y cuenta cuántos caen en cada intervalo. El apilamiento por color permite observar la composición de los niveles didácticos.

- Una concentración a la izquierda indica muchos SKU con cobertura estimada corta.
- Una cola larga a la derecha indica algunos SKU con cobertura muy alta; no necesariamente es un error.
- Las líneas verticales marcan los umbrales configurados.
- El tope de 180 días solo recorta la coordenada para hacer legible la figura; no modifica `cobertura_dias`.

No se debe concluir “hay desabasto” solo porque existan SKU bajo 15 días. La lectura correcta es “hay SKU que, bajo esta ventana y estos umbrales didácticos, merecen revisión prioritaria”.

**Pregunta:** ¿por qué puede existir cobertura muy alta?  


### Actividad en equipos

### Preguntas

- ¿La mayoría de los SKUs parece concentrarse debajo o arriba de los umbrales?
- ¿Qué información puede explicar una cobertura muy alta además de “mucho inventario”?
- ¿Por qué no sería correcto eliminar automáticamente los valores extremos?

- Filtrar los SKUs válidos que tienen 180 días o más de cobertura
- Calculamos totales y porcentaje sobre el valor total del inventario cruzado
- Tabla con los 10 productos que más dinero tienen detenido
- Gráfica con zoom en la zona operativa (0 a 60 días)

## Gráfica 2 — ¿Qué productos críticos tienen mayor impacto comercial?

`barplot` permite comparar una métrica agregada ya calculada. Aquí se muestran los productos con **cobertura crítica** ordenados por venta neta reciente.


In [ ]:
# Filtra la tabla para quedarse solo con los artículos que tienen menos de UMBRAL_CRITICO días de inventario.
top_criticos = (
    tabla_riesgo.loc[tabla_riesgo["nivel_riesgo"].eq("Crítico")]
    .sort_values("venta_neta", ascending=False).head(15)
    .sort_values("venta_neta")
)
if top_criticos.empty:
    print("No hay SKU críticos con los parámetros actuales.")
else:
    fig, ax = plt.subplots(figsize=(12, 7))
    sns.barplot(data=top_criticos, x="venta_neta", y="nombre_articulo",
                hue="marca", dodge=False, legend=False, palette="Reds_r", ax=ax)
    ax.set(title=f"Top 15 SKU críticos por venta neta reciente ({DAYS} días)",
           xlabel="Venta neta registrada en la ventana", ylabel="Producto")
    plt.tight_layout()
    plt.show()

### Cómo leer el barplot — interpretación

Cada barra representa la venta neta registrada para un SKU clasificado como crítico. El eje vertical identifica el producto; el horizontal permite comparar magnitudes. Se muestran los 15 SKU con mayor venta neta dentro del grupo crítico.

**¿Por qué ordenar por venta neta y no solo por menor cobertura?** Porque la visualización busca combinar una señal de cobertura corta con una dimensión comercial. No significa que la venta neta sea margen, rentabilidad o venta perdida.

Un artículo con cobertura baja y venta neta alta puede ameritar una revisión prioritaria, pero la decisión también requiere validar inventario disponible real, pedidos en tránsito, lead time, margen, sustitutos y compromisos.

## Gráfica 3 — Matriz exploratoria de demanda y existencias

`scatterplot` revela relaciones entre dos variables cuantitativas. Cada punto es un SKU con correspondencia en inventario. El tamaño representa venta neta y el color el nivel de riesgo.

Para que unos pocos valores extremos no oculten al resto, se limita la vista al percentil 99 de ambos ejes. La tabla completa conserva los datos sin recorte.


In [ ]:
scatter_df = tabla_riesgo.loc[
    tabla_riesgo["match_inventario"] & tabla_riesgo["demanda_diaria"].gt(0)
].copy()
if scatter_df.empty:
    print("No hay SKU comparables con demanda positiva.")
else:
    #Recorte al percentil 99 (quantile(0.99)): En ventas e inventarios casi siempre existe 1 o 2
    # productos extremos (por ejemplo, un artículo de altísima rotación o un lote gigante en almacén) 
    # que empujan los ejes hacia arriba y hacia la derecha, amontonando el 99% restante de los puntos
    # en una esquina. Al excluir el 1% más extremo de ambos ejes, la gráfica hace un "auto-zoom" 
    # sobre la zona donde opera la inmensa mayoría del negocio.

    xmax = scatter_df["demanda_diaria"].quantile(0.99)
    ymax = scatter_df["existencias"].quantile(0.99)
    scatter_vista = scatter_df[
        (scatter_df["demanda_diaria"] <= xmax) &
        (scatter_df["existencias"] <= ymax)
    ].copy()
    fig, ax = plt.subplots(figsize=(11, 7))
    sns.scatterplot(
        data=scatter_vista, x="demanda_diaria", y="existencias",
        hue="nivel_riesgo", size="venta_neta", sizes=(25, 450), alpha=.72,
        hue_order=["Crítico", "Vigilar", "Saludable (regla didáctica)"],
        palette={"Crítico": "#C62828", "Vigilar": "#F9A825",
                 "Saludable (regla didáctica)": "#2E7D32"}, ax=ax,
    )
    ax.set(title="Demanda diaria observada vs existencias al corte por SKU",
           xlabel="Unidades vendidas por día (promedio de la ventana)",
           ylabel="Existencias al corte")

    #Como la leyenda combina dos cosas (los colores del nivel de riesgo y los distintos tamaños de círculo
    #  para la venta neta), se vuelve bastante alta. bbox_to_anchor=(1.02, 1) la coloca fuera del área
    #  del gráfico (a la derecha) para que no tape ninguna burbuja importante.
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left",
              title="Nivel / venta neta")
    plt.tight_layout()
    plt.show()

### Cómo leer el scatterplot — interpretación

Cada punto representa un SKU comparable:
- Eje X: promedio de unidades vendidas por día durante la ventana.
- Eje Y: existencias registradas al corte.
- Color: nivel exploratorio.
- Tamaño: venta neta registrada.

 Gráfico de dispersión multivariado (gráfico de burbujas) que cruza la velocidad de venta contra el inventario disponible, añadiendo el impacto económico de cada producto mediante el tamaño de los puntos. Es una visualización ejecutiva ideal para priorizar decisiones, ya que distingue rápidamente entre un producto crítico que deja mucho dinero y otro que apenas aporta ingresos.

**Lectura por cuadrantes:** puntos hacia la derecha tienen mayor velocidad de venta observada; puntos hacia arriba tienen más existencias. Los puntos abajo y a la derecha pueden combinar pocas existencias con mayor actividad reciente y, por ello, aparecer como candidatos a revisión.

**¿Una nube ascendente prueba que el inventario responde a la demanda?** No. Es una asociación descriptiva entre dos variables. No establece causalidad ni demuestra que el inventario se haya planeado correctamente.

**Nota:** se excluyen los percentiles superiores al 99 para que los valores extremos no compriman la nube principal. Es un filtro de visualización, no una eliminación de registros del análisis.

## Gráfica 4 — ¿Qué marcas concentran el riesgo?

Un `heatmap` es útil para comparar una matriz. Mostramos **porcentajes por marca**, no solo conteos: esto evita que las marcas con catálogos más grandes parezcan riesgosas únicamente por tener más SKUs.


In [ ]:
marca_base = tabla_riesgo.dropna(subset=["marca"]).copy()
#Crea una tabla cruzada donde cada fila es una marca y cada columna es un nivel de riesgo.
# El parámetro normalize="index" divide cada celda entre el total de SKUs de esa misma marca
marca_riesgo = pd.crosstab(
    marca_base["marca"], marca_base["nivel_riesgo"], normalize="index"
).mul(100)
marca_riesgo = marca_riesgo.reindex(columns=orden_riesgo, fill_value=0)

#En lugar de graficar decenas o cientos de marcas irrelevantes, agrupa por marca, 
# suma su venta_neta y extrae con .nlargest(12) los nombres de las 12 marcas que más ingresos generan.
top_marcas = (
    marca_base.groupby("marca")["venta_neta"].sum().nlargest(12).index
)
marca_riesgo_top = marca_riesgo.reindex(top_marcas).fillna(0)

fig, ax = plt.subplots(figsize=(11, 7))
sns.heatmap(
    marca_riesgo_top[orden_riesgo], annot=True, fmt=".1f", cmap="YlOrRd",
    linewidths=.4, cbar_kws={"label": "% de SKU dentro de la marca"}, ax=ax
)
ax.set(title="Composición porcentual de niveles exploratorios por marca",
       xlabel="Clasificación didáctica", ylabel="Marca")
plt.tight_layout()
plt.show()

### Cómo leer el heatmap — interpretación

Cada fila es una marca y cada celda muestra el porcentaje de SKU de esa marca que cae en una categoría. Los porcentajes se calculan por fila, por lo que cada marca suma aproximadamente 100% (puede haber diferencias pequeñas por redondeo).

**¿Por qué usar porcentajes y no conteos?** 



| Pregunta | Respuesta esperada |
|---|---|
| ¿Por qué agregar antes del `merge`? | Para alinear la granularidad y evitar multiplicación de filas y doble conteo. |
| ¿Qué mide la cobertura de llaves? | Coincidencia de identificadores únicos, no integridad de datos ni demanda cubierta. |
| ¿Por qué separar no-match y stock cero? | Porque no encontrar una llave no demuestra que el inventario sea cero. |
| ¿Qué significa cobertura de 12 días? | Bajo la tasa media observada en la ventana, el stock equivaldría a 12 días; no es una predicción garantizada. |
| ¿Por qué 15 y 30 días? | Son umbrales didácticos configurables, no políticas validadas. |
| ¿Por qué excluir SKU sin ventas del cálculo? | La división no es interpretable cuando la demanda diaria es cero o negativa; se reportan aparte. |
| ¿Qué representa el tamaño del punto? | Venta neta registrada en el periodo, no utilidad ni pérdida por falta de stock. |
| ¿Por qué usar porcentajes en el heatmap? | Para comparar la composición relativa entre marcas de distinto tamaño. |
| ¿Un outlier es un error? | No necesariamente; primero se investiga su origen y significado. |
| ¿Cuál es el resultado final? | Una lista de SKU para revisión analítica, no una orden de compra. |


In [ ]:
# 1. Filtramos SKUs válidos y topamos a 180 días para evitar distorsión por outliers
violin_df = tabla_riesgo.loc[
    tabla_riesgo["match_inventario"] & tabla_riesgo["cobertura_dias"].notna()
].copy()
violin_df["cobertura_grafica"] = violin_df["cobertura_dias"].clip(upper=180)

# 2. Seleccionamos las 8 marcas principales (excluimos las que casi no tienen match)
top_8_marcas = (
    violin_df.groupby("marca")["venta_neta"]
    .sum()
    .nlargest(8)
    .index
)
violin_top = violin_df[violin_df["marca"].isin(top_8_marcas)]

# 3. Generamos la gráfica de violín horizontal (más fácil de leer con nombres de marca)
fig, ax = plt.subplots(figsize=(11, 6))
sns.violinplot(
    data=violin_top,
    x="cobertura_grafica",
    y="marca",
    order=top_8_marcas,
    hue="marca",
    palette="deep",
    legend=False,
    cut=0,          # Evita que la curva baje de 0 días o pase de 180
    inner="box",    # Dibuja el boxplot (mediana y cuartiles) dentro de cada violín
    ax=ax
)

# Líneas de umbral de negocio
ax.axvline(UMBRAL_CRITICO, color="#C62828", linestyle="--", linewidth=1.5,
           label=f"Crítico ({UMBRAL_CRITICO} días)")
ax.axvline(UMBRAL_VIGILAR, color="#F9A825", linestyle="--", linewidth=1.5,
           label=f"Vigilar ({UMBRAL_VIGILAR} días)")

ax.set(
    title="Distribución de días de cobertura por marca (Top 8 en ventas)",
    xlabel="Cobertura estimada (días; tope visual 180)",
    ylabel="Marca"
)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


Expone tres comportamientos completamente distintos entre las 8 marcas más importantes en ventas, algo que el heatmap anterior ocultaba al agrupar todo en categorías fijas.


1. Marcas con "embudo hacia el sobreinventario" (AIDITE, AG, VITA, ANAX DENT y CREATION)

• Qué se ve: Estos cinco violines tienen forma de aguja muy delgada a la izquierda y se ensanchan bruscamente al llegar a los 160–180 días. Además, su punto blanco (mediana) y su caja negra interior están aplastados completamente contra el borde derecho (180).
• Qué significa: Más del 75% del catálogo de estas marcas tiene al menos 6 meses de inventario acumulado. Aunque AIDITE, AG y VITA tienen una cola delgada que cruza las líneas roja y amarilla (uno o dos productos aislados en riesgo), el verdadero problema financiero de estas marcas es el exceso de stock. En el caso de ANAX DENT y CREATION, ni siquiera tienen productos por debajo de los 45 y 90 días respectivamente.

2. El contraste de rotación saludable pero dispersa (JENSE)

• Qué se ve: El violín azul de JENSE es completamente distinto a los demás: es un cilindro ancho y parejo desde casi 1 día hasta 180 días, con su mediana (punto blanco) ubicada alrededor de los 104 días y su caja negra (50% central de los datos) extendiéndose desde ~45 hasta 180 días.
• Qué significa: JENSE es la marca con mayor dinamismo real de rotación; sus productos están repartidos a lo largo de todo el espectro de cobertura en lugar de estar estancados en 180 días. Sin embargo, como toda su distribución está recorrida hacia la izquierda, es también la única marca con un bloque significativo de artículos cruzando las líneas de Vigilar (30 días) y Crítico (15 días).

3. Comportamiento de MEDIT y ROKO (alta dispersión sin quiebre inmediato)

• Qué se ve:
	• MEDIT (morado) tiene un cuerpo ancho que arranca justo después de la línea crítica (~16 días) y va ensanchándose hacia la derecha, aunque su mediana sigue pegada en 180 días y su rango intercuartílico (barra negra) baja hasta los ~66 días.
	• ROKO (café) muestra un bloque rectangular grueso que empieza en ~60 días, con mediana en ~120 días.
• Qué significa: Ninguna de las dos tiene productos en estado crítico (< 15 días). En el caso de ROKO, tiene una cobertura muy holgada (mínimo 2 meses de stock en todos sus productos), pero importante recordar que, como tiene pocos SKUs totales, el algoritmo de densidad dibuja una caja bastante uniforme entre su valor mínimo (60) y máximo (180).
